# Tests

> MCP tools for generating tests, stubs, and aggregating TODOs/BUGs

In [ ]:
#| default_exp tools.tests

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple

import re
import ast
from pathlib import Path

from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.re import TODO_CODE_PATTERN, TODO_MD_PATTERN, BUG_PATTERN
from nbdev_mcp.utils.paths import (
    nbs_dir, settings_dict, resolve_selector, iter_notebooks,
    read_nb, write_nb, abs_module_for_nb,
)
from nbdev_mcp.utils.nb import join_source, find_default_exp, has_export
from nbdev_mcp.utils.rich import render_table, render_panel

## Test Generation

In [ ]:
#| export
def generate_pytests(
    project: Optional[str] = None,
    dry_run: bool = False,
    long_test_eval_false: bool = True
) -> Dict[str, Any]:
    """Generate pytest-compatible modules from notebook test cells.
    
    Extracts test code from notebooks (cells with asserts or test functions)
    and generates pytest modules under tests/.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    dry_run : bool
        If True, don't write files, just report what would be done.
    long_test_eval_false : bool
        If True, add '#| eval: false' to long test cells (40+ lines).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'generated' list and 'modified_cells' list.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    nbs_root = nbs_dir(p)
    outdir = p / 'tests'
    outdir.mkdir(exist_ok=True)
    
    generated: List[str] = []
    modified_cells: List[Tuple[str, int]] = []
    
    for nb in iter_notebooks(p):
        if not str(nb).startswith(str(nbs_root)) or nb.name == 'index.ipynb':
            continue
        
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        test_funcs: List[str] = []
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Check if cell has test content
            if 'assert' not in src and not re.search(r'\bpytest\b|^def\s+test_', src, flags=re.MULTILINE):
                continue
            
            body_lines = [ln for ln in src.splitlines() if not ln.strip().startswith('#|')]
            if not body_lines:
                continue
            
            func_name = f'test_nb_cell_{i:03d}'
            test_funcs.append('\n'.join([
                f'def {func_name}():',
                *[f'    {ln}' if ln.strip() else '' for ln in body_lines],
                ''
            ]))
            
            # Mark long cells as eval: false
            if long_test_eval_false and len(body_lines) >= 40 and '#| eval: false' not in src:
                cell['source'] = ('#| eval: false\n' + src).splitlines(True)
                modified_cells.append((str(nb.relative_to(p)), i))
        
        if test_funcs:
            py = [
                f'# Auto-generated from {nb.relative_to(p)}\n',
                f'import {lib}.{mod} as m\n\n'
            ]
            py.extend(test_funcs)
            
            out_file = outdir / f"test_{mod.replace('.', '_')}.py"
            if not dry_run:
                out_file.write_text('\n'.join(py), encoding='utf-8')
            generated.append(str(out_file.relative_to(p)))
            
            if long_test_eval_false and modified_cells and not dry_run:
                write_nb(nb, data)
    
    if generated:
        rows = [[g] for g in generated]
        pretty = render_table('generated pytest files', ['path'], rows)
    else:
        pretty = render_panel('pytest generator', 'No tests found to extract.')
    
    return {
        'ok': True,
        'generated': generated,
        'modified_cells': modified_cells,
        'dry_run': dry_run,
        'pretty': pretty
    }

In [ ]:
#| export
def scaffold_test_utils(project: Optional[str] = None) -> Dict[str, Any]:
    """Create the standard nbs/99_tests/00_utils.ipynb notebook.
    
    Creates a testing utilities notebook with shared helpers and mocks
    for use across the test suite.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'created' path or 'message' if already exists.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nb_dir = nbs_dir(p) / '99_tests'
    nb_dir.mkdir(parents=True, exist_ok=True)
    nb_path = nb_dir / '00_utils.ipynb'
    
    if nb_path.exists():
        return {'ok': True, 'message': f'Exists: {nb_path.relative_to(p)}'}
    
    nb = {
        'cells': [
            {
                'cell_type': 'markdown',
                'metadata': {},
                'source': ['# Test Utilities\n', '> Helpers and mocks for tests\n']
            },
            {
                'cell_type': 'code',
                'metadata': {},
                'source': ['#| default_exp tests.utils\n'],
                'outputs': [],
                'execution_count': None
            },
            {
                'cell_type': 'code',
                'metadata': {},
                'source': [
                    '#| export\n',
                    'def make_example_data(n: int = 3):\n',
                    '    "Return a simple list of ints for testing."\n',
                    '    return list(range(n))\n'
                ],
                'outputs': [],
                'execution_count': None
            }
        ],
        'metadata': {
            'kernelspec': {'name': 'python3', 'language': 'python'}
        },
        'nbformat': 4,
        'nbformat_minor': 5
    }
    
    write_nb(nb_path, nb)
    return {'ok': True, 'created': str(nb_path.relative_to(p))}

## Stub Generation

In [ ]:
#| export
def generate_stubs(
    project: Optional[str] = None,
    out_path: Optional[str] = None,
    include_methods: bool = True
) -> Dict[str, Any]:
    """Generate .pyi stub files for exported symbols.
    
    Scans notebooks for exported functions and classes, generating
    type stub files for better IDE support.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    out_path : str, optional
        Custom output path for the stub file.
    include_methods : bool
        If True, include method signatures in class stubs.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'out_file' path.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    stubs_dir = p / 'stubs'
    stubs_dir.mkdir(exist_ok=True)
    out = Path(out_path) if out_path else stubs_dir / f'{lib}.pyi'
    
    modules: Dict[str, List[str]] = {}
    
    def add_stub(mod: str, text: str) -> None:
        modules.setdefault(mod, []).append(text)
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            if not has_export(src):
                continue
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            for node in tree.body:
                if isinstance(node, ast.FunctionDef):
                    sig = ast.get_source_segment(src, node.args) or '(…)'
                    add_stub(mod, f'def {node.name}{sig}: ...  # from {nb.name}#cell{i}')
                elif isinstance(node, ast.AsyncFunctionDef):
                    sig = ast.get_source_segment(src, node.args) or '(…)'
                    add_stub(mod, f'async def {node.name}{sig}: ...  # from {nb.name}#cell{i}')
                elif isinstance(node, ast.ClassDef):
                    bases = [ast.get_source_segment(src, b) or 'object' for b in node.bases]
                    add_stub(mod, f"class {node.name}({', '.join(bases) if bases else 'object'}): ...  # from {nb.name}#cell{i}")
                    if include_methods:
                        for sub in node.body:
                            if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                                sig = ast.get_source_segment(src, sub.args) or '(…)'
                                prefix = 'async def' if isinstance(sub, ast.AsyncFunctionDef) else 'def'
                                add_stub(mod, f'    {prefix} {sub.name}{sig}: ...')
    
    lines = [
        '# Auto-generated stubs — do not edit\n',
        f'# Package: {lib}\n\n',
        'from __future__ import annotations\n',
        'from typing import *\n\n'
    ]
    for mod in sorted(modules):
        lines.append(f'# --- {lib}.{mod} ---\n')
        for item in modules[mod]:
            lines.append(item + '\n')
        lines.append('\n')
    
    out.write_text(''.join(lines), encoding='utf-8')
    
    pretty = render_panel('stubs', f'Stub written: {out}')
    
    return {'ok': True, 'out_file': str(out.relative_to(p)), 'pretty': pretty}

## Comment Aggregation

In [ ]:
#| export
def aggregate_todos(
    project: Optional[str] = None,
    out_file: str = 'TODOs.md'
) -> Dict[str, Any]:
    """Aggregate TODO comments from notebooks into a Markdown file.
    
    Scans all notebooks for TODO comments and collects them into
    a single Markdown table.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    out_file : str
        Output filename (default: TODOs.md).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'count' of TODOs and 'out_file' path.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    rows: List[Tuple[str, int, int, str]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        rel = str(nb.relative_to(p))
        
        for i, cell in enumerate(data.get('cells', [])):
            src = join_source(cell.get('source', []))
            
            if cell.get('cell_type') == 'code':
                for j, ln in enumerate(src.splitlines(), 1):
                    m = TODO_CODE_PATTERN.search(ln)
                    if m:
                        rows.append((rel, i, j, m.group(0).strip()))
            else:
                for j, ln in enumerate(src.splitlines(), 1):
                    if 'TODO' in ln.upper():
                        m = TODO_MD_PATTERN.search(ln)
                        if m:
                            rows.append((rel, i, j, m.group(0).strip()))
    
    outp = p / out_file
    lines = [
        '# TODOs\n\n',
        '| Notebook | Cell | Line | Comment |\n',
        '|---|---:|---:|---|\n'
    ]
    for rel, i, j, txt in rows:
        lines.append(f"| {rel} | {i} | {j} | {txt.replace('|', '\\|')} |\n")
    outp.write_text(''.join(lines), encoding='utf-8')
    
    return {'ok': True, 'count': len(rows), 'out_file': str(outp.relative_to(p))}

In [ ]:
#| export
def aggregate_bugs(
    project: Optional[str] = None,
    out_file: str = 'BUGs.md'
) -> Dict[str, Any]:
    """Aggregate BUG comments from notebooks into a Markdown file.
    
    Scans all notebooks for BUG comments and collects them into
    a single Markdown table.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    out_file : str
        Output filename (default: BUGs.md).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'count' of BUGs and 'out_file' path.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    rows: List[Tuple[str, int, int, str]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        rel = str(nb.relative_to(p))
        
        for i, cell in enumerate(data.get('cells', [])):
            src = join_source(cell.get('source', []))
            for j, ln in enumerate(src.splitlines(), 1):
                if BUG_PATTERN.search(ln):
                    rows.append((rel, i, j, ln.strip()))
    
    outp = p / out_file
    lines = [
        '# BUGs\n\n',
        '| Notebook | Cell | Line | Comment |\n',
        '|---|---:|---:|---|\n'
    ]
    for rel, i, j, txt in rows:
        lines.append(f"| {rel} | {i} | {j} | {txt.replace('|', '\\|')} |\n")
    outp.write_text(''.join(lines), encoding='utf-8')
    
    return {'ok': True, 'count': len(rows), 'out_file': str(outp.relative_to(p))}

## MCP Registration

In [ ]:
#| export
def add_tests_tools(mcp: FastMCP) -> None:
    """Attach generation tools to the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(generate_pytests)
    mcp.add_tool(scaffold_test_utils)
    mcp.add_tool(generate_stubs)
    mcp.add_tool(aggregate_todos)
    mcp.add_tool(aggregate_bugs)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()